<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Higgs-Audio_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Higgs Audio v3 TTS — Expressive Multilingual Speech with Voice Cloning

State-of-the-art conversational TTS by Boson AI. 4B-parameter autoregressive model that produces expressive, multi-speaker dialogue-grade speech across 100+ languages. Zero-shot voice cloning from a reference clip. Outperforms Qwen3-TTS-1.7B on the Boson-Higgs-Multilingual benchmark.

### Quick Start
1. Connect to a GPU runtime (A100 / L4 recommended; 16 GB+ VRAM)
2. Run Steps 1–4 in order — no token, no sign-up needed
3. Open the Gradio UI link from Step 4 and start generating

| GPU | VRAM | Status |
|-----|------|--------|
|-----|------|--------|
| A100 | 40 GB | Recommended (fastest) |
|-----|------|--------|
| L4 | 24 GB | Recommended |
|-----|------|--------|
| T4 | 16 GB | Works (tighter on VRAM) |

**License notice:** Higgs Audio v3 weights are released under the *Boson Higgs Audio v3 Research and Non-Commercial License*. Outputs are for research and personal use only. Production / commercial / hosted use requires a separate license from Boson AI. The model card explicitly prohibits voice cloning without consent, impersonation, fraud, election deception, and biometric surveillance.

This notebook uses the [`multimodalart/higgs-audio-v3-tts-4b-transformers`](https://huggingface.co/multimodalart/higgs-audio-v3-tts-4b-transformers) trust-remote-code port — same weights, but loads with plain 🤗 Transformers (no SGLang server required).

In [ ]:
# @title STEP 1 — Install Dependencies
# @markdown Installs 🤗 Transformers (>= 5.5 required), torchaudio, Gradio, and Moonshine (for auto-transcribing reference clips).

import os, sys, subprocess
import torch

if not torch.cuda.is_available():
    raise SystemExit('No GPU detected. Connect a GPU runtime: Runtime → Change runtime type → T4 / L4 / A100')

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'[GPU] {gpu_name} — {vram_gb:.1f} GB')

# transformers >= 5.5 is required by the trust_remote_code port
print('[pip] Upgrading transformers (needs >= 5.5) ...')
subprocess.run(['pip', 'install', '-q', '-U', 'transformers>=5.5'], check=True)
print(f'  transformers {__import__("transformers").__version__}')

# torchaudio for the trust-remote-code example; gradio for the UI; soundfile for wav I/O
print('[pip] Installing torchaudio, gradio, soundfile, librosa ...')
subprocess.run(['pip', 'install', '-q', '-U', 'torchaudio', 'gradio==5.33.0', 'soundfile', 'librosa', 'numpy', 'tqdm==4.66.5'], check=True)

# Moonshine — tiny ASR model used to auto-transcribe reference clips.
# Kept on CPU so it doesn't eat VRAM from the main TTS model.
print('[pip] Installing Moonshine ASR (for auto-transcribing reference clips) ...')
subprocess.run(['pip', 'install', '-q', '-U', 'transformers', 'UsefulSensors/moonshine-base @ git+https://huggingface.co/UsefulSensors/moonshine-base'], check=False)
subprocess.run(['pip', 'install', '-q', '-U', 'transformers', '--upgrade-strategy', 'only-if-needed'], check=True)
print('[pip] Done.')

os.makedirs('/content/audio_out', exist_ok=True)
os.makedirs('/content/ref_audio', exist_ok=True)
print('[Setup] /content/audio_out and /content/ref_audio ready.')

In [ ]:
# @title STEP 2 — Pre-cache Models
# @markdown Downloads the TTS weights, the v2 audio codec, and Moonshine ASR to the local Colab cache.

# @markdown Reuses the Drive-cached weights from `TTS_Model_Loader.ipynb` if present,
# @markdown otherwise downloads to the default ~/.cache/huggingface (in-session only).
import pathlib
try:
    from google.colab import drive
    if not pathlib.Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive', force_remount=False)
    CACHE_ROOT = pathlib.Path('/content/drive/MyDrive/AEI_TTS_Cache')
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ.setdefault('HF_HOME', str(CACHE_ROOT / 'huggingface'))
    print(f'[Cache] using Drive at {CACHE_ROOT}')
except Exception:
    print('[Cache] Drive not available, using default ~/.cache/huggingface')


import os
from huggingface_hub import snapshot_download

REPOS = {
    'TTS (transformers port)': 'multimodalart/higgs-audio-v3-tts-4b-transformers',
    'Audio codec (Higgs v2)':  'bosonai/higgs-audio-v2-tokenizer',
    'ASR (Moonshine)':         'UsefulSensors/moonshine-base',
}

print('[Download] Pre-caching model weights ...')
for tag, repo in REPOS.items():
    print(f'  → {repo}  ({tag})')
    snapshot_download(repo)
print('[Download] All weights cached.')

In [ ]:
# @title STEP 3 — Core Functions (lazy model loading)

# @markdown Defines the inference wrappers and a CPU-only ASR helper for auto-transcribing reference clips.



import os, time, gc

import numpy as np

import torch

import torchaudio

import soundfile as sf



REPO = 'multimodalart/higgs-audio-v3-tts-4b-transformers'

CODEC_REPO = 'bosonai/higgs-audio-v2-tokenizer'

ASR_REPO = 'UsefulSensors/moonshine-base'

ASR_SAMPLE_RATE = 16000



CACHE = {}



def load_tts():

    if 'tts' in CACHE:

        return CACHE['tts']

    from transformers import AutoModelForCausalLM, AutoTokenizer

    print(f'[Load] {REPO} (bf16) ...')

    t0 = time.time()

    tok = AutoTokenizer.from_pretrained(REPO)

    m = AutoModelForCausalLM.from_pretrained(REPO, trust_remote_code=True, dtype=torch.bfloat16).to('cuda').eval()

    m.get_audio_codec()  # pre-load the fp32 codec

    print(f'[Load] TTS ready in {time.time()-t0:.1f}s — sample_rate={m.config.sample_rate} Hz')

    CACHE['tts'] = (m, tok)

    return CACHE['tts']



def load_asr():

    if 'asr' in CACHE:

        return CACHE['asr']

    from transformers import AutoProcessor, MoonshineForConditionalGeneration

    print(f'[Load] {ASR_REPO} (CPU, for auto-transcribing reference clips) ...')

    proc = AutoProcessor.from_pretrained(ASR_REPO)

    m = MoonshineForConditionalGeneration.from_pretrained(ASR_REPO).eval()

    CACHE['asr'] = (proc, m)

    return CACHE['asr']



def transcribe_ref(path: str) -> str:

    """CPU-only auto-transcription of a reference clip via Moonshine."""

    if not path:

        return ''

    proc, m = load_asr()

    data, sr = sf.read(path, dtype='float32', always_2d=True)

    wav = torch.from_numpy(data).mean(dim=1)  # mono [L]

    if sr != ASR_SAMPLE_RATE:

        wav = torchaudio.functional.resample(wav, orig_freq=sr, new_freq=ASR_SAMPLE_RATE)

    inputs = proc(wav.numpy(), sampling_rate=ASR_SAMPLE_RATE, return_tensors='pt')

    with torch.no_grad():

        tokens = m.generate(**inputs)

    return proc.decode(tokens[0], skip_special_tokens=True).strip()



def _load_ref_tensor(path: str):

    """Return (wav_tensor[L] or [C,L], sample_rate)."""

    wav, sr = torchaudio.load(path)  # [C, L]

    if wav.shape[0] > 1:

        wav = wav.mean(dim=0, keepdim=True)

    return wav.squeeze(0), sr  # [L], sr



def synth(text: str,

          ref_audio_path: str | None = None,

          ref_text: str = '',

          temperature: float = 0.7,

          top_p: float = 0.95,

          top_k: int = 50,

          max_new_tokens: int = 2048,

          seed: int = -1,

          out_name: str = 'higgs.wav') -> tuple[str, int]:

    m, tok = load_tts()

    text = (text or '').strip()

    if not text:

        raise RuntimeError('Text is empty.')

    if seed is not None and int(seed) >= 0:

        torch.manual_seed(int(seed))



    kwargs = dict(

        max_new_tokens=int(max_new_tokens),

        temperature=float(temperature),

        top_p=float(top_p) if float(top_p) < 1.0 else None,

        top_k=int(top_k) if int(top_k) > 0 else None,

    )



    if ref_audio_path:

        ref_wav, ref_sr = _load_ref_tensor(ref_audio_path)

        kwargs['reference_audio'] = ref_wav

        kwargs['reference_sample_rate'] = int(ref_sr)

        if ref_text and ref_text.strip():

            kwargs['reference_text'] = ref_text.strip()



    wav = m.generate_speech(text, tok, **kwargs)

    if wav.numel() == 0:

        raise RuntimeError('Generation produced no audio — try again or adjust the text.')



    out_path = os.path.join('/content/audio_out', out_name)

    sr = m.config.sample_rate

    sf.write(out_path, wav.numpy(), sr)

    return out_path, sr



print('[Core] Ready. Use Step 4 for the UI, Step 6 for quick test, Step 7 for batch.')

In [ ]:
# @title STEP 4 — Gradio UI
# @markdown Interactive web app for zero-shot TTS + voice cloning. Click the `.gradio.live` link to open.

import sys
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass
import gradio as gr

def _err(msg):
    return None, f'❌ {msg}'

def ui_synth(text, ref_audio, ref_text, auto_transcribe, temperature, top_p, top_k, max_new_tokens, seed, progress=gr.Progress()):
    if not text or not text.strip():
        return _err('Type some text first.')
    try:
        progress(0.05, desc='Loading TTS model...')
        load_tts()
        # Auto-transcribe if user uploaded audio and asked for it
        effective_ref_text = (ref_text or '').strip()
        if ref_audio is not None and auto_transcribe and not effective_ref_text:
            progress(0.2, desc='Auto-transcribing reference clip (CPU)...')
            try:
                effective_ref_text = transcribe_ref(ref_audio)
                yield None, f'✍️ Auto-transcribed: "{effective_ref_text}"\n⏳ Generating...'
            except Exception as e:
                print(f'[ASR] failed: {e} — continuing without transcript')
                effective_ref_text = ''
        progress(0.4, desc='Synthesizing...')
        path, sr = synth(
            text.strip(),
            ref_audio_path=ref_audio,
            ref_text=effective_ref_text,
            temperature=temperature, top_p=top_p, top_k=top_k,
            max_new_tokens=max_new_tokens, seed=seed,
        )
        progress(1.0, desc='Done.')
        yield path, f'✅ Generated at {sr} Hz'
    except Exception as e:
        yield None, f'❌ {e}'

def ui_transcribe(ref_audio):
    if not ref_audio:
        return ''
    try:
        return transcribe_ref(ref_audio)
    except Exception as e:
        return f'[ASR error] {e}'

with gr.Blocks(title='Higgs Audio v3 TTS', theme=gr.themes.Soft()) as demo:
    gr.Markdown(f'# \ud83c\udf4b Higgs Audio v3 TTS\n**Model:** `bosonai/higgs-audio-v3-tts-4b` (4B, bf16) \u00b7 **GPU:** {gpu_name} ({vram_gb:.0f} GB)\n\nZero-shot TTS and voice cloning. 100+ languages. Optionally auto-transcribes your reference clip with Moonshine ASR.')
    with gr.Row():
        with gr.Column():
            text = gr.Textbox(
                label='Text to synthesize',
                value='Higgs Audio version three, running on plain transformers in Google Colab. It speaks, not just reads.',
                lines=4,
                placeholder='Type what you want the voice to say…',
            )
            ref_audio = gr.Audio(
                label='Reference voice (optional — upload for voice cloning)',
                type='filepath', info="5-30s of clean single-speaker audio. The transcript (auto or manual) significantly improves fidelity.",
                )
            with gr.Row():
                auto_tx = gr.Checkbox(label='Auto-transcribe reference clip with Moonshine ASR', value=True)
                tx_btn = gr.Button('Transcribe now', size='sm')
            ref_text = gr.Textbox(
                label='Reference transcript (auto-filled if you enable auto-transcribe — improves cloning fidelity)',
                lines=2,
            )
            with gr.Accordion('Advanced sampling', open=False):
                temperature = gr.Slider(0.0, 1.5, value=0.7, step=0.05, label='Temperature', info="Lower = more deterministic, higher = more varied. 0.7 is a good default.")
                top_p = gr.Slider(0.1, 1.0, value=0.95, step=0.01, label='Top-p (1.0 = off)', info="Nucleus sampling: only consider tokens whose cumulative probability is below this. 0.9 = off.")
                top_k = gr.Slider(0, 200, value=50, step=1, label='Top-k (0 = off)', info="Only consider top K tokens at each step. 50 is a good default; 0 = disabled.")
                max_new_tokens = gr.Slider(64, 4096, value=2048, step=64, label='Max new tokens', info="Maximum audio length to generate. 0 = until the model says EOS.")
                seed = gr.Number(value=-1, label='Seed (-1 = random)', precision=0)
            btn = gr.Button('Generate speech', variant='primary', size='lg')
        with gr.Column():
            output_audio = gr.Audio(label='Generated speech', type='filepath')
            status = gr.Markdown('')
    gr.Examples(
        examples=[
            ['The quick brown fox jumps over the lazy dog.', None, ''],
            ['\u00a1Hola! Este es Higgs Audio, hablando espa\u00f1ol con un poco de emoci\u00f3n.', None, ''],
            ['<|emotion:amusement|><|prosody:expressive_high|>Wait, that was kind of hilarious. <|sfx:laughter|>Haha, no, seriously, I was not ready for that.', None, ''],
        ],
        inputs=[text, ref_audio, ref_text],
        label='Try a sample (the emotion / sfx tags are passed through to the model as text)',
    )
    btn.click(
        ui_synth,
        inputs=[text, ref_audio, ref_text, auto_tx, temperature, top_p, top_k, max_new_tokens, seed],
        outputs=[output_audio, status],
    )
    tx_btn.click(ui_transcribe, inputs=[ref_audio], outputs=[ref_text])
    gr.Markdown('**Tips:** For voice cloning, 3–10 seconds of clean, single-speaker reference audio gives the best results. Reference transcript (auto or manual) materially improves fidelity. The inline emotion / style / sfx tags shown in the model card (`<|emotion:amusement|>`, etc.) are passed through as text in this port — see the upstream SGLang docs for the full reference.')

from IPython.display import clear_output as _clear
_clear()
demo.queue(max_size=8, concurrency_limit=2).launch(
    share=True, show_error=True, server_name="0.0.0.0", server_port=7860,
)
demo.load(lambda: "🎙️ Higgs Audio v3 ready. Pick an example below, or type your own text.", outputs=[_model_load_status])


In [ ]:
# @title Step 5 — Keep Alive
# @markdown Prevents Colab from disconnecting during long generation sessions. Run anytime after launching the UI.

import threading, time
def _keep():
    while True:
        time.sleep(60)
        try:
            import requests
            requests.get('https://www.google.com', timeout=5)
        except Exception:
            pass
threading.Thread(target=_keep, daemon=True).start()
print('[Keep-Alive] Running. Stop cell to disable.')

In [ ]:
# @title Step 6 — Quick Test (one-off inference, no UI)
# @markdown Run a single TTS call from the notebook. Useful for scripting or quick checks.

TEXT = "Hello! This is Higgs Audio v3 running in Google Colab." # @param {type:"string"}
REF_AUDIO = "" # @param {type:"string"}
REF_TEXT = "" # @param {type:"string"}
AUTO_TRANSCRIBE = True # @param {type:"boolean"}
TEMPERATURE = 0.7 # @param {type:"number"}
TOP_P = 0.95 # @param {type:"number"}
TOP_K = 50 # @param {type:"number"}
MAX_NEW_TOKENS = 2048 # @param {type:"integer"}
SEED = -1 # @param {type:"integer"}

effective_ref_text = (REF_TEXT or '').strip()
if REF_AUDIO and AUTO_TRANSCRIBE and not effective_ref_text:
    try:
        effective_ref_text = transcribe_ref(REF_AUDIO)
        print(f'[ASR] auto-transcript: "{effective_ref_text}"')
    except Exception as e:
        print(f'[ASR] failed: {e} — continuing without transcript')
        effective_ref_text = ''

path, sr = synth(
    TEXT,
    ref_audio_path=REF_AUDIO or None,
    ref_text=effective_ref_text,
    temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
    max_new_tokens=MAX_NEW_TOKENS, seed=SEED,
)
print(f'[Done] {path} ({sr} Hz)')
from IPython.display import Audio, display
display(Audio(path))

In [ ]:
# @title Step 7 — Batch Synthesis
# @markdown Generate audio for a list of texts. Each line in `BATCH` becomes one output file.

BATCH_REF_AUDIO = '' # @param {type:"string"}
BATCH_REF_TEXT = '' # @param {type:"string"}
BATCH_TEMPERATURE = 0.7 # @param {type:"number"}
BATCH_TOP_P = 0.95 # @param {type:"number"}
BATCH_TOP_K = 50 # @param {type:"number"}
BATCH_MAX_NEW_TOKENS = 2048 # @param {type:"integer"}
BATCH_SEED = -1 # @param {type:"integer"}

BATCH = """\
The quick brown fox jumps over the lazy dog.
She sells seashells by the seashore.
To be, or not to be: that is the question.
It was the best of times, it was the worst of times.
All happy families are alike; each unhappy family is unhappy in its own way.
"""

lines = [l.strip() for l in BATCH.strip().splitlines() if l.strip()]
out_dir = '/content/audio_out/batch'
os.makedirs(out_dir, exist_ok=True)
effective_ref_text = (BATCH_REF_TEXT or '').strip()

for i, line in enumerate(lines):
    try:
        print(f'[{i+1}/{len(lines)}] {line[:60]}{"..." if len(line) > 60 else ""}')
        path, sr = synth(
            line,
            ref_audio_path=BATCH_REF_AUDIO or None,
            ref_text=effective_ref_text,
            temperature=BATCH_TEMPERATURE, top_p=BATCH_TOP_P, top_k=BATCH_TOP_K,
            max_new_tokens=BATCH_MAX_NEW_TOKENS, seed=BATCH_SEED,
            out_name=f'{i:03d}.wav',
        )
        # synth already writes to /content/audio_out/<name>; move to batch subdir
        os.rename(path, os.path.join(out_dir, f'{i:03d}.wav'))

    except Exception as e:
        print(f"  -> EXCEPTION: {e}")
        continue
print(f'\n[Done] {len(lines)} files written to {out_dir}')